# Data Integration

This notebook integrates the cleaned weather and air-quality datasets for Delhi.

Both datasets contain hourly observations for the year 2025 and share the `time` column as their common key.

### Objectives

- Load the cleaned weather dataset
- Load the cleaned air-quality dataset
- Verify their structure and time coverage
- Check timestamp alignment
- Merge both datasets using the `time` column
- Validate the merged dataset
- Remove redundant columns created during the merge
- Save the unified environmental dataset

### Output

The final integrated dataset will contain weather, air-quality, and time-related variables and will serve as the foundation for subsequent climate-impact analysis.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [4]:
weather_path = Path("../data/processed/weather_delhi_cleaned.csv")

weather_df = pd.read_csv(weather_path)

print("Weather dataset loaded successfully.")
print("Shape:", weather_df.shape)

weather_df.head()

Weather dataset loaded successfully.
Shape: (8760, 15)


,time,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,cloud_cover,wind_speed_10m,wind_gusts_10m,year,month,day,hour,day_of_year,season,rain_flag
0,2025-01-01 00:00:00,8.3,100,7.4,0.0,99,2.8,4.7,2025,1,1,0,1,Winter,0
1,2025-01-01 01:00:00,8.0,100,7.0,0.0,100,2.9,5.4,2025,1,1,1,1,Winter,0
2,2025-01-01 02:00:00,7.8,100,6.5,0.0,100,4.7,10.1,2025,1,1,2,1,Winter,0
3,2025-01-01 03:00:00,8.0,99,6.5,0.0,100,5.8,12.6,2025,1,1,3,1,Winter,0
4,2025-01-01 04:00:00,7.7,98,6.1,0.0,100,6.3,13.0,2025,1,1,4,1,Winter,0


In [5]:
air_quality_path = Path(
    "../data/processed/air_quality_delhi_cleaned.csv"
)

air_quality_df = pd.read_csv(air_quality_path)

print("Air-quality dataset loaded successfully.")
print("Shape:", air_quality_df.shape)

air_quality_df.head()

Air-quality dataset loaded successfully.
Shape: (8760, 13)


,time,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,year,month,day,hour,day_of_year,season
0,2025-01-01 00:00:00,185.8,188.6,1645.0,56.7,49.7,31.0,2025,1,1,0,1,Winter
1,2025-01-01 01:00:00,174.6,177.4,1551.0,44.8,39.7,39.0,2025,1,1,1,1,Winter
2,2025-01-01 02:00:00,164.4,166.7,1478.0,34.6,31.5,46.0,2025,1,1,2,1,Winter
3,2025-01-01 03:00:00,156.5,158.8,1418.0,26.7,26.0,52.0,2025,1,1,3,1,Winter
4,2025-01-01 04:00:00,149.5,151.8,1379.0,20.6,22.3,57.0,2025,1,1,4,1,Winter


In [6]:
weather_df["time"] = pd.to_datetime(
    weather_df["time"]
)

air_quality_df["time"] = pd.to_datetime(
    air_quality_df["time"]
)

In [7]:
print("WEATHER DATA")
print("Start:", weather_df["time"].min())
print("End:", weather_df["time"].max())
print("Observations:", weather_df["time"].nunique())

print("\nAIR QUALITY DATA")
print("Start:", air_quality_df["time"].min())
print("End:", air_quality_df["time"].max())
print("Observations:", air_quality_df["time"].nunique())

WEATHER DATA
Start: 2025-01-01 00:00:00
End: 2025-12-31 23:00:00
Observations: 8760

AIR QUALITY DATA
Start: 2025-01-01 00:00:00
End: 2025-12-31 23:00:00
Observations: 8760


In [8]:
weather_times = set(weather_df["time"])
air_quality_times = set(air_quality_df["time"])

weather_only = weather_times - air_quality_times
air_quality_only = air_quality_times - weather_times

print(
    "Timestamps present only in weather:",
    len(weather_only)
)

print(
    "Timestamps present only in air quality:",
    len(air_quality_only)
)

Timestamps present only in weather: 0
Timestamps present only in air quality: 0


In [9]:
print(
    "Weather duplicate timestamps:",
    weather_df["time"].duplicated().sum()
)

print(
    "Air-quality duplicate timestamps:",
    air_quality_df["time"].duplicated().sum()
)

Weather duplicate timestamps: 0
Air-quality duplicate timestamps: 0


In [10]:
overlapping_columns = set(weather_df.columns).intersection(
    set(air_quality_df.columns)
)

print("Overlapping columns:")
print(overlapping_columns)

Overlapping columns:
{'time', 'day_of_year', 'season', 'month', 'hour', 'year', 'day'}


In [11]:
air_quality_for_merge = air_quality_df[
    [
        "time",
        "pm2_5",
        "pm10",
        "carbon_monoxide",
        "nitrogen_dioxide",
        "sulphur_dioxide",
        "ozone"
    ]
].copy()

In [12]:
environmental_df = pd.merge(
    weather_df,
    air_quality_for_merge,
    on="time",
    how="inner",
    validate="one_to_one"
)

In [13]:
print(
    "Merged dataset shape:",
    environmental_df.shape
)

Merged dataset shape: (8760, 21)


In [14]:
environmental_df.head()

,time,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,cloud_cover,wind_speed_10m,wind_gusts_10m,year,month,day,hour,day_of_year,season,rain_flag,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone
0,2025-01-01 00:00:00,8.3,100,7.4,0.0,99,2.8,4.7,2025,1,1,0,1,Winter,0,185.8,188.6,1645.0,56.7,49.7,31.0
1,2025-01-01 01:00:00,8.0,100,7.0,0.0,100,2.9,5.4,2025,1,1,1,1,Winter,0,174.6,177.4,1551.0,44.8,39.7,39.0
2,2025-01-01 02:00:00,7.8,100,6.5,0.0,100,4.7,10.1,2025,1,1,2,1,Winter,0,164.4,166.7,1478.0,34.6,31.5,46.0
3,2025-01-01 03:00:00,8.0,99,6.5,0.0,100,5.8,12.6,2025,1,1,3,1,Winter,0,156.5,158.8,1418.0,26.7,26.0,52.0
4,2025-01-01 04:00:00,7.7,98,6.1,0.0,100,6.3,13.0,2025,1,1,4,1,Winter,0,149.5,151.8,1379.0,20.6,22.3,57.0


In [15]:
environmental_df.columns.tolist()

['time',
 'temperature_2m',
 'relative_humidity_2m',
 'apparent_temperature',
 'precipitation',
 'cloud_cover',
 'wind_speed_10m',
 'wind_gusts_10m',
 'year',
 'month',
 'day',
 'hour',
 'day_of_year',
 'season',
 'rain_flag',
 'pm2_5',
 'pm10',
 'carbon_monoxide',
 'nitrogen_dioxide',
 'sulphur_dioxide',
 'ozone']

In [16]:
missing_values = environmental_df.isnull().sum()

print(missing_values)

print(
    "\nTotal missing values:",
    environmental_df.isnull().sum().sum()
)

time                    0
temperature_2m          0
relative_humidity_2m    0
apparent_temperature    0
precipitation           0
cloud_cover             0
wind_speed_10m          0
wind_gusts_10m          0
year                    0
month                   0
day                     0
hour                    0
day_of_year             0
season                  0
rain_flag               0
pm2_5                   0
pm10                    0
carbon_monoxide         0
nitrogen_dioxide        0
sulphur_dioxide         0
ozone                   0
dtype: int64

Total missing values: 0


In [17]:
print(
    "Duplicate rows:",
    environmental_df.duplicated().sum()
)

print(
    "Duplicate timestamps:",
    environmental_df["time"].duplicated().sum()
)

Duplicate rows: 0
Duplicate timestamps: 0


In [18]:
is_sorted = environmental_df["time"].is_monotonic_increasing

print(
    "Dataset sorted chronologically:",
    is_sorted
)

Dataset sorted chronologically: True


In [19]:
environmental_df = (
    environmental_df
    .sort_values("time")
    .reset_index(drop=True)
)

In [20]:
environmental_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   time                  8760 non-null   datetime64[us]
 1   temperature_2m        8760 non-null   float64       
 2   relative_humidity_2m  8760 non-null   int64         
 3   apparent_temperature  8760 non-null   float64       
 4   precipitation         8760 non-null   float64       
 5   cloud_cover           8760 non-null   int64         
 6   wind_speed_10m        8760 non-null   float64       
 7   wind_gusts_10m        8760 non-null   float64       
 8   year                  8760 non-null   int64         
 9   month                 8760 non-null   int64         
 10  day                   8760 non-null   int64         
 11  hour                  8760 non-null   int64         
 12  day_of_year           8760 non-null   int64         
 13  season                8760 no

In [21]:
environmental_df.describe().T

,count,mean,min,25%,50%,75%,max,std
time,8760,2025-07-02 11:30:00,2025-01-01 00:00:00,2025-04-02 05:45:00,2025-07-02 11:30:00,2025-10-01 17:15:00,2025-12-31 23:00:00,NaN
temperature_2m,8760.0,25.275342,7.7,20.0,26.6,30.5,43.7,7.463091
relative_humidity_2m,8760.0,60.617922,4.0,42.0,63.0,80.0,100.0,23.657999
apparent_temperature,8760.0,26.892831,5.8,19.1,28.9,34.5,46.6,9.271445
precipitation,8760.0,0.081735,0.0,0.0,0.0,0.0,12.0,0.530325
cloud_cover,8760.0,34.272717,0.0,0.0,7.0,88.0,100.0,41.845766
wind_speed_10m,8760.0,6.640685,0.0,4.2,6.1,8.6,25.8,3.4176
wind_gusts_10m,8760.0,17.578573,0.7,11.2,16.2,23.0,71.6,8.730237
year,8760.0,2025.0,2025.0,2025.0,2025.0,2025.0,2025.0,0.0
month,8760.0,6.526027,1.0,4.0,7.0,10.0,12.0,3.448048


In [22]:
environmental_df["season"].value_counts()

season
Monsoon         2928
Summer          2208
Winter          2160
Post-Monsoon    1464
Name: count, dtype: int64

In [23]:
print("========== INTEGRATION VALIDATION ==========")

print("\nRows:", len(environmental_df))
print("Columns:", len(environmental_df.columns))

print(
    "\nMissing values:",
    environmental_df.isnull().sum().sum()
)

print(
    "Duplicate rows:",
    environmental_df.duplicated().sum()
)

print(
    "Duplicate timestamps:",
    environmental_df["time"].duplicated().sum()
)

print(
    "Chronologically sorted:",
    environmental_df["time"].is_monotonic_increasing
)

print(
    "Unique timestamps:",
    environmental_df["time"].nunique()
)

========== INTEGRATION VALIDATION ==========

Rows: 8760
Columns: 21

Missing values: 0
Duplicate rows: 0
Duplicate timestamps: 0
Chronologically sorted: True
Unique timestamps: 8760
